In [ ]:
import pandas as pd
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler

torch.cuda.empty_cache()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cpu')

In [ ]:
## ***** PREPROCESSING DATA booooooooorrriiiingggg😒😒🙄 ***** ##

def preprocess(df, is_train=True):
    df = df.drop(columns=['PassengerId', 'Name'])

    # fill null rows with the median instead of dropping
    non_binary = [col for col in df.columns if df[col].nunique() > 2 and df[col].dtype in ['float64', 'int64']]
    for col in non_binary:
        df[col] = df[col].fillna(df[col].median())

    # fill categorical rows with the mode
    binary_cols = [col for col in df.columns if df[col].nunique() == 2]
    for col in binary_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    cat_cols = ['HomePlanet', 'Destination', 'Cabin']
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    # only drop nulls for train target column
    if is_train:
        df.dropna(subset=['Transported'], inplace=True)

    df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = df['Cabin'].str.split('/', expand=True)
    df = df.drop(columns=['Cabin'])
    df['Cabin_Num'] = df['Cabin_Num'].astype(int)

    df[["CryoSleep", "VIP"]] = df[["CryoSleep", "VIP"]].astype(int)

    df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Cabin_Deck", "Cabin_Side"])

    dummy_cols = [col for col in df.columns if df[col].dtype == 'bool']
    df[dummy_cols] = df[dummy_cols].astype(int)

    non_binary = [col for col in df.columns if df[col].nunique() > 2 and df[col].dtype in ['float64', 'int64']]
    
    if is_train:
        df[non_binary] = scaler.fit_transform(df[non_binary])  # fit AND transform
    else:
        df[non_binary] = scaler.transform(df[non_binary])  # only transform

    return df

scaler = StandardScaler()
df_train = preprocess(pd.read_csv('train.csv'), is_train=True)
df_test = preprocess(pd.read_csv('test.csv'), is_train=False)
df_train

In [ ]:
X_train_pre = df_train.drop(columns=['Transported']).values
Y_train_pre = df_train['Transported'].values

In [ ]:
X_train = torch.tensor(X_train_pre, dtype=torch.float32).to(device)
Y_train = torch.tensor(Y_train_pre, dtype=torch.float32).to(device)
print(torch.cuda.is_available())  

#can skip this since it a comparatively small dataset but it is a good habit to manage it in batches for training faster and also shuffle the datapoints --
# --for each epoch preventing the model to learn order patterns
dataset = TensorDataset(X_train, Y_train)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

Y_train

In [ ]:
## *** LEESSS GOOOO 💥💥🦅 *** ##

class Model(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 1),
            nn.Sigmoid()

            # nn.Linear(input_size, 256),
            # nn.BatchNorm1d(256),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.3),

            # nn.Linear(256, 128),
            # nn.BatchNorm1d(128),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.3),

            # nn.Linear(128, 64),
            # nn.BatchNorm1d(64),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.2),

            # nn.Linear(64, 32),
            # nn.LeakyReLU(0.01),

            # nn.Linear(32, 1),
            # nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)
    
input_size = X_train.shape[1]
model = Model(input_size).to(device)

In [ ]:
## *** LOCK-IN ✌️ *** ##

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)

epochs=400

for epoch in range(epochs):
    model.train()
    total_loss=0
    correct=0
    total=0

    for X_batch, Y_batch in dataloader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        output = model(X_batch).squeeze()
        loss = criterion(output, Y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        predicted = (output > 0.5).float()
        correct += (predicted == Y_batch).sum().item()
        total += Y_batch.size(0)

    if (epoch%20 == 0):
            avgloss = total_loss/len(dataloader)
            accuracy = correct / total * 100
            print(f'Epoch {epoch}/{epochs}, Loss: {avgloss:.4f}, Accuracy: {accuracy:.2f}%')


In [ ]:
passenger_ids = pd.read_csv('test.csv')['PassengerId']

model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(df_test.values.astype('float32'), dtype=torch.float32).to(device)
    predictions = model(X_test_tensor).squeeze()
    predictions = (predictions > 0.5).cpu().numpy().astype(bool)

df_submit = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': predictions
})

df_submit.to_csv('submission.csv', index=False)
print(df_submit.head())
print(df_submit.shape)